In [0]:
import numpy as np
import pandas as pd

woe_cols = [c for c in spark.table("workspace.default.lc_test_scored").columns if c.endswith("_woe")]
pdf = spark.table("workspace.default.lc_test_scored").select(woe_cols).toPandas()

corr = pdf.corr().values
vif_vals = np.diag(np.linalg.inv(corr))

vif = pd.DataFrame({"feature": woe_cols, "VIF": vif_vals}).sort_values("VIF", ascending=False)
print(vif.to_string(index=False))

                    feature      VIF
              grade_num_woe 1.303148
            term_months_woe 1.188242
                inc_bin_woe 1.121679
                dti_bin_woe 1.085803
     home_ownership_idx_woe 1.079156
verification_status_idx_woe 1.071432
            purpose_idx_woe 1.021610


In [0]:
import pandas as pd
import numpy as np

el = spark.table("workspace.default.lc_test_el").toPandas()
el = el.sort_values("pd_prediction").reset_index(drop=True)   # best risk first
total_n = len(el)

rows = []
for q in np.arange(0.10, 1.001, 0.05):     # approve the safest q fraction of applicants
    n = int(total_n * q)
    book = el.iloc[:n]
    income = book["interest_income"].sum()
    loss   = book["EL"].sum()
    rows.append({
        "approval_rate": round(q, 2),
        "loans_approved": n,
        "pd_cutoff": round(book["pd_prediction"].max(), 4),
        "income_$m": round(income/1e6, 1),
        "exp_loss_$m": round(loss/1e6, 1),
        "net_contribution_$m": round((income - loss)/1e6, 1),
        "approved_EL_rate_%": round(100*loss/book["EAD"].sum(), 2),
    })

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7582540964112451>, line 4
      1 import pandas as pd
      2 import numpy as np
----> 4 el = spark.table("workspace.default.lc_test_el").toPandas()
      5 el = el.sort_values("pd_prediction").reset_index(drop=True)   # best risk first
      6 total_n = len(el)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1943, in DataFrame.toPandas(self)
   1941 def toPandas(self) -> "PandasDataFrameLike":
   1942     query = self._plan.to_proto(self._session.client)
-> 1943     pdf, ei = self._session.client.to_pandas(query, self._plan.observations)
   1944     self._execution_info = ei
   1945     return pdf

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1386, in SparkConnectClient.to_pandas(self, plan, observations)
   1375 # Get all related configs in

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|default |lc_analytical       |false      |
|default |lc_features         |false      |
|default |lc_scorecard_points |false      |
|default |lc_test_scored      |false      |
|default |lc_test_scored_final|false      |
|default |lc_test_woe         |false      |
|default |lc_train_woe        |false      |
+--------+--------------------+-----------+



In [0]:
print("scored_final:", spark.table("workspace.default.lc_test_scored_final").columns)
print("analytical:  ", spark.table("workspace.default.lc_analytical").columns)

scored_final: ['id', 'pd_prediction', 'is_bad', 'funded_amnt', 'int_rate', 'total_rec_prncp', 'recoveries', 'loan_status', 'grade']
analytical:   ['id', 'loan_amnt', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths

In [0]:
from pyspark.sql.functions import col, expr, regexp_replace

PORTFOLIO_LGD = 0.8903   # Day 7 realised LGD on charged-off loans — reused, not refitted

df = (spark.table("workspace.default.lc_test_scored_final")
      # defensive casts — your CSV had garbage bleeding into numeric cols (Day 4 lesson)
      .withColumn("ead",  expr("try_cast(funded_amnt AS DOUBLE)"))
      .withColumn("rate", expr("try_cast(regexp_replace(cast(int_rate AS STRING), '%', '') AS DOUBLE)"))
      .withColumn("EAD", col("ead"))
      .withColumn("EL",  col("pd_prediction") * PORTFOLIO_LGD * col("ead"))
      .withColumn("interest_income", col("ead") * (col("rate")/100.0)))

df.select("id","pd_prediction","EAD","EL","interest_income","grade") \
  .write.mode("overwrite").saveAsTable("workspace.default.lc_test_el")
print("rows:", df.count())

rows: 225639


In [0]:
from pyspark.sql.functions import sum as _sum, count as _cnt, when
a = df.agg(
        _sum("EL").alias("el"), _sum("EAD").alias("ead"),
        _sum("interest_income").alias("inc"),
        _cnt(when(col("interest_income").isNull(), 1)).alias("null_inc")
    ).collect()[0]
print(f"rows {df.count():,} | EL ${a['el']:,.0f} | EL rate {a['el']/a['ead']:.2%} "
      f"| income ${a['inc']:,.0f} | null income rows {a['null_inc']}")

rows 225,639 | EL $598,114,433 | EL rate 18.35% | income $466,781,870 | null income rows 0


In [0]:
import pandas as pd
import numpy as np

el = spark.table("workspace.default.lc_test_el").toPandas()
el = el.sort_values("pd_prediction").reset_index(drop=True)   # best risk first
total_n = len(el)

rows = []
for q in np.arange(0.10, 1.001, 0.05):     # approve the safest q fraction of applicants
    n = int(total_n * q)
    book = el.iloc[:n]
    income = book["interest_income"].sum()
    loss   = book["EL"].sum()
    rows.append({
        "approval_rate": round(q, 2),
        "loans_approved": n,
        "pd_cutoff": round(book["pd_prediction"].max(), 4),
        "income_$m": round(income/1e6, 1),
        "exp_loss_$m": round(loss/1e6, 1),
        "net_contribution_$m": round((income - loss)/1e6, 1),
        "approved_EL_rate_%": round(100*loss/book["EAD"].sum(), 2),
    })

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

 approval_rate  loans_approved  pd_cutoff  income_$m  exp_loss_$m  net_contribution_$m  approved_EL_rate_%
          0.10           22563     0.0621       21.8         14.4                  7.5                4.47
          0.15           33845     0.0759       31.8         22.5                  9.2                4.95
          0.20           45127     0.0918       45.4         33.5                 11.8                5.56
          0.25           56409     0.1064       60.2         46.4                 13.8                6.20
          0.30           67691     0.1204       75.5         60.7                 14.8                6.82
          0.35           78973     0.1330       91.8         77.0                 14.7                7.44
          0.40           90255     0.1459      109.4         95.4                 14.0                8.06
          0.45          101537     0.1588      127.3        115.2                 12.1                8.67
          0.50          112819     0.

In [0]:
best = sweep.loc[sweep["net_contribution_$m"].idxmax()]
print(best)

approval_rate              0.3000
loans_approved         67691.0000
pd_cutoff                  0.1204
income_$m                 75.5000
exp_loss_$m               60.7000
net_contribution_$m       14.8000
approved_EL_rate_%         6.8200
Name: 4, dtype: float64


In [0]:
spark.createDataFrame(sweep) \
     .write.mode("overwrite").saveAsTable("workspace.default.lc_approval_sweep")